<a href="https://colab.research.google.com/github/mayura-andrew/ml-experiments/blob/main/AngryBird_Maze_game_RL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
!pip install gymnasium numpy

In [72]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import random

class AngryBirdsGridEnv(gym.Env):
    # Required for rendering output
    metadata = {"render_modes": ["human"]}

    def __init__(self, render_mode=None):
        super().__init__()
        self.render_mode = render_mode
        self.grid_size = 5

        # Action Space: 0 = Move Forward, 1 = Turn Left, 2 = Turn Right
        self.action_space = spaces.Discrete(3)

        # Observation Space: A 5x5 matrix representing the board
        # 0: Empty, 1: Wall, 2: Bird, 3: Pig
        self.observation_space = spaces.Box(
            low=0, high=4, shape=(self.grid_size, self.grid_size + 1,), dtype=np.int32
        )

        # Initialize state variables
        self.grid = np.zeros((self.grid_size, self.grid_size), dtype=np.int32)
        self.bird_pos = [0, 0]
        self.bird_dir = 0 # 0: North, 1: East, 2: South, 3: West

    def _get_obs(self):
        # Flatten the grid into a 1D array and append the bird's direction
        flat_grid = self.grid.flatten()
        return np.append(flat_grid, self.bird_dir).astype(np.float32)


    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        # Start with a clean board
        self.grid = np.zeros((self.grid_size, self.grid_size), dtype=np.int32)

        # Define our test cases (different level layouts)
        # Format: walls = list of (row, col), pig = [row, col], bird = [row, col], dir = direction
        levels = [
            # Case 0: The Original Obstacle
            {
                "walls": [(1, 1), (1, 2), (1, 3), (3, 1)],
                "pig": [4, 4], "bird": [0, 0], "dir": 1 # Facing East
            },
            # Case 1: The "S" Curve Corridor
            {
                "walls": [(1, 0), (1, 1), (1, 2), (3, 2), (3, 3), (3, 4)],
                "pig": [4, 0], "bird": [0, 4], "dir": 2 # Facing South
            },
            # Case 2: The Corner Trap
            {
                "walls": [(2, 2), (2, 3), (3, 2)],
                "pig": [4, 4], "bird": [0, 0], "dir": 1 # Facing East
            },
             # Case 3: Pig in the Middle
            {
                "walls": [(1, 1), (3, 3), (1, 3), (3, 1)],
                "pig": [2, 2], "bird": [4, 0], "dir": 0 # Facing North
            }
        ]

        # Randomly pick a level for this episode
        current_level = random.choice(levels)

        # 1. Place the walls
        for r, c in current_level["walls"]:
            self.grid[r, c] = 1

        # 2. Place the Pig
        self.pig_pos = current_level["pig"]
        self.grid[self.pig_pos[0], self.pig_pos[1]] = 3

        # 3. Place the Bird and set its direction
        self.bird_pos = current_level["bird"]
        self.bird_dir = current_level["dir"]
        self.grid[self.bird_pos[0], self.bird_pos[1]] = 2

        return self._get_obs(), {}

    def step(self, action):
        reward = -1  # Default penalty for taking a step
        terminated = False
        truncated = False

        # 1. Handle Turning
        if action == 1:  # Turn Left
            self.bird_dir = (self.bird_dir - 1) % 4
        elif action == 2:  # Turn Right
            self.bird_dir = (self.bird_dir + 1) % 4

        # 2. Handle Moving Forward
        elif action == 0:
            # Calculate intended next position based on current direction
            # Directions: 0=North, 1=East, 2=South, 3=West
            dr, dc = 0, 0
            if self.bird_dir == 0:   dr = -1
            elif self.bird_dir == 1: dc = 1
            elif self.bird_dir == 2: dr = 1
            elif self.bird_dir == 3: dc = -1

            new_r = self.bird_pos[0] + dr
            new_c = self.bird_pos[1] + dc

            # 3. Collision Detection
            # Check if out of bounds
            if new_r < 0 or new_r >= self.grid_size or new_c < 0 or new_c >= self.grid_size:
                reward = -10
                terminated = True
            else:
                target_cell = self.grid[new_r, new_c]

                # Check if hit a wall
                if target_cell == 1:
                    reward = -10
                    terminated = True
                # Check if hit the pig
                elif target_cell == 3:
                    reward = 100
                    terminated = True

                # If safe, update the grid and bird position
                if not terminated:
                    # Clear old position
                    self.grid[self.bird_pos[0], self.bird_pos[1]] = 0
                    # Update to new position
                    self.bird_pos = [new_r, new_c]
                    # Set bird on new position
                    self.grid[self.bird_pos[0], self.bird_pos[1]] = 2

        # Update bird direction on the grid even if we just turned
        if not terminated:
            self.grid[self.bird_pos[0], self.bird_pos[1]] = 2

        return self._get_obs(), reward, terminated, truncated, {}


    def render(self):
        if self.render_mode == "human":
            # Map our matrix integers to visual characters
            # ^, >, v, < will show which way the bird is facing
            dir_chars = ['^', '>', 'v', '<']

            print("-" * (self.grid_size * 2 + 1))
            for r in range(self.grid_size):
                row_str = "|"
                for c in range(self.grid_size):
                    val = self.grid[r, c]
                    if val == 0: row_str += ". "  # Empty space
                    elif val == 1: row_str += "# " # Wall
                    elif val == 2: row_str += f"{dir_chars[self.bird_dir]} " # Bird
                    elif val == 3: row_str += "P " # Pig
                print(row_str[:-1] + "|")
            print("-" * (self.grid_size * 2 + 1))
            print("\n")

In [66]:
if __name__ == "__main__":
    env = AngryBirdsGridEnv(render_mode="human")

    # Initialize the board
    observation, info = env.reset()

    # Draw it to the terminal
    print("Initial State:")
    env.render()

Initial State:
-----------
|> . . . .|
|. . . . .|
|. . # # .|
|. . # . .|
|. . . . P|
-----------




In [71]:
if __name__ == "__main__":
    env = AngryBirdsGridEnv(render_mode="human")
    observation, info = env.reset()

    print("=== Angry Birds RL Environment ===")
    env.render()

    terminated = False
    truncated = False
    total_reward = 0

    print("Controls: 'w' = Move Forward, 'a' = Turn Left, 'd' = Turn Right, 'q' = Quit")

    while not (terminated or truncated):
        user_input = input("Enter action: ").lower().strip()

        if user_input == 'q':
            print("Quitting test loop.")
            break

        # Map human input to our discrete action space
        if user_input == 'w':
            action = 0
        elif user_input == 'a':
            action = 1
        elif user_input == 'd':
            action = 2
        else:
            print("Invalid input. Please use w, a, d, or q.")
            continue

        # Step the environment forward
        observation, reward, terminated, truncated, info = env.step(action)
        total_reward += reward

        # Display the results of the action
        print(f"\nAction taken: {user_input} | Reward received: {reward} | Total Reward: {total_reward}")
        env.render()

        # Win/Loss condition check
        if terminated:
            if reward == 100:
                print("Success! You reached the pig!")
            else:
                print("Game Over! You hit a wall or went out of bounds.")

=== Angry Birds RL Environment ===
-----------
|> . . . .|
|. # # # .|
|. . . . .|
|. # . . .|
|. . . . P|
-----------


Controls: 'w' = Move Forward, 'a' = Turn Left, 'd' = Turn Right, 'q' = Quit


KeyboardInterrupt: Interrupted by user

In [36]:
!pip install torch

In [58]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DQN(nn.Module):
    def __init__(self, grid_size=5, num_actions=3):
        super(DQN, self).__init__()

        # Calculate the total number of inputs (5 * 5 = 25)
        self.input_dim = (grid_size * grid_size) + 1

        # Layer 1: Input to Hidden
        self.fc1 = nn.Linear(self.input_dim, 64)

        # Layer 2: Hidden to Hidden
        self.fc2 = nn.Linear(64, 64)

        # Layer 3: Hidden to Output (Q-values for each action)
        self.out = nn.Linear(64, num_actions)

    def forward(self, x):
        # Flatten the 2D grid into a 1D vector
        # x = x.view(-1, self.input_dim)

        # Pass through layers with ReLU activation for non-linearity
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        # Output layer (no activation function here, we want raw Q-values)
        return self.out(x)

In [73]:
import random
from collections import deque
import numpy as np

class ReplayMemory:
    def __init__(self, capacity=10000):
        # A deque automatically removes the oldest items when it reaches capacity
        self.memory = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        # Save a single step (a "transition") to memory
        self.memory.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        # Randomly sample a batch of transitions for training
        # This breaks the correlation of sequential steps
        batch = random.sample(self.memory, batch_size)

        # Unpack the batch into separate arrays for PyTorch
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states), np.array(actions), np.array(rewards),
                np.array(next_states), np.array(dones))

    def __len__(self):
        # Allows us to check how full the memory is using len(memory)
        return len(self.memory)

In [60]:

import torch
import torch.optim as optim
import random
import numpy as np

class DQNAgent:
    def __init__(self, state_dim=25, action_dim=3, lr=0.001):
        self.action_dim = action_dim
        self.gamma = 0.99 # How much we care about future rewards

        # Epsilon parameters for Exploration vs. Exploitation
        self.epsilon = 1.0       # Start 100% random
        self.epsilon_min = 0.05  # Never go below 5% random
        self.epsilon_decay = 0.995 # Decay rate per episode

        # 1. Initialize the Brain (Neural Network)
        self.policy_net = DQN(grid_size=5, num_actions=action_dim)

        # 2. Initialize the Optimizer (Adam is standard for Deep Learning)
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)

        # 3. Initialize the Loss Function (Mean Squared Error)
        self.loss_fn = torch.nn.MSELoss()

        # 4. Initialize the Memory
        self.memory = ReplayMemory(capacity=10000)

    def select_action(self, state):
        # Epsilon-Greedy Logic
        if random.random() < self.epsilon:
            # EXPLORE: Pick a random number between 0 and 2
            return random.randrange(self.action_dim)
        else:
            # EXPLOIT: Ask the Neural Network
            # Convert the numpy array state into a PyTorch Tensor
            state_tensor = torch.FloatTensor(state).unsqueeze(0)

            with torch.no_grad(): # We are just predicting, not training here
                q_values = self.policy_net(state_tensor)

            # Return the index of the highest Q-value (0, 1, or 2)
            return q_values.argmax().item()

    def optimize_model(self, batch_size=32):
        # Don't try to train if we don't have enough memories yet
        if len(self.memory) < batch_size:
            return

        # Grab a random batch of memories
        states, actions, rewards, next_states, dones = self.memory.sample(batch_size)

        # Convert everything to PyTorch Tensors
        states = torch.FloatTensor(states)
        actions = torch.LongTensor(actions).unsqueeze(1)
        rewards = torch.FloatTensor(rewards).unsqueeze(1)
        next_states = torch.FloatTensor(next_states)
        dones = torch.FloatTensor(dones).unsqueeze(1) # 1 if game over, 0 if continuing

        # --- THE MACHINE LEARNING HAPPENS HERE ---

        # A. What did our network predict the Q-value was for the action we took?
        current_q_values = self.policy_net(states).gather(1, actions)

        # B. What SHOULD the Q-value have been? (Bellman Equation)
        with torch.no_grad():
            # Find the best possible move in the NEXT state
            max_next_q = self.policy_net(next_states).max(1)[0].unsqueeze(1)
            # If done is 1, (1-done) becomes 0, zeroing out future rewards because the game ended.
            target_q_values = rewards + (self.gamma * max_next_q * (1 - dones))

        # C. Calculate the Loss (Error)
        loss = self.loss_fn(current_q_values, target_q_values)

        # D. Backpropagation (Update the neural network weights to reduce the error)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    def update_epsilon(self):
        # Slowly decrease epsilon after each game to stop exploring and start exploiting
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

In [61]:
if __name__ == "__main__":
    # 1. Initialize Environment (Render mode OFF for fast training)
    env = AngryBirdsGridEnv(render_mode=None)

    # 2. Initialize the Agent
    agent = DQNAgent(state_dim=26, action_dim=3)

    # 3. Training Parameters
    total_episodes = 10000  # Increased from 5000
    batch_size = 32

    print("Starting Training Phase...")

    for episode in range(total_episodes):
        # Reset the environment for a new game
        state, _ = env.reset()

        total_reward = 0
        done = False
        step_count = 0

        # The game loop (runs until the bird wins or dies)
        while not done:
            step_count += 1

            # A. Agent chooses an action (Explore vs. Exploit)
            action = agent.select_action(state)

            # B. The environment reacts
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # C. Save the experience to memory
            agent.memory.push(state, action, reward, next_state, done)

            # D. Train the neural network on past memories
            agent.optimize_model(batch_size)

            # E. Move to the next state
            state = next_state
            total_reward += reward

            # Safety catch: prevent infinite loops if the agent just spins in circles
            if step_count > 50:
                break

        # Decrease epsilon so the agent explores less and exploits more over time
        agent.update_epsilon()

        # Print progress every 50 episodes
        if (episode + 1) % 50 == 0:
            print(f"Episode {episode + 1}/{total_episodes} | "
                  f"Total Reward: {total_reward:4} | "
                  f"Epsilon: {agent.epsilon:.3f} | "
                  f"Memory Size: {len(agent.memory)}")

    print("\nTraining Complete!")

    # Save the trained brain to a file
    torch.save(agent.policy_net.state_dict(), "angry_birds_dqn.pth")
    print("Model saved to 'angry_birds_dqn.pth'")

Starting Training Phase...
Episode 50/10000 | Total Reward:  -28 | Epsilon: 0.778 | Memory Size: 320
Episode 100/10000 | Total Reward:  -14 | Epsilon: 0.606 | Memory Size: 655
Episode 150/10000 | Total Reward:  -12 | Epsilon: 0.471 | Memory Size: 948
Episode 200/10000 | Total Reward:  -14 | Epsilon: 0.367 | Memory Size: 1227
Episode 250/10000 | Total Reward:  -15 | Epsilon: 0.286 | Memory Size: 1489
Episode 300/10000 | Total Reward:  -14 | Epsilon: 0.222 | Memory Size: 1719
Episode 350/10000 | Total Reward:  -15 | Epsilon: 0.173 | Memory Size: 1965
Episode 400/10000 | Total Reward:  -12 | Epsilon: 0.135 | Memory Size: 2192
Episode 450/10000 | Total Reward:  -14 | Epsilon: 0.105 | Memory Size: 2405
Episode 500/10000 | Total Reward:  -14 | Epsilon: 0.082 | Memory Size: 2604
Episode 550/10000 | Total Reward:  -11 | Epsilon: 0.063 | Memory Size: 2844
Episode 600/10000 | Total Reward:  -12 | Epsilon: 0.050 | Memory Size: 3044
Episode 650/10000 | Total Reward:  -14 | Epsilon: 0.050 | Memory 

In [74]:
import time
import torch

if __name__ == "__main__":
    print("Loading trained neural network...")

    # 1. Initialize Environment WITH rendering turned on
    env = AngryBirdsGridEnv(render_mode="human")

    # 2. Initialize the Brain and load the saved weights
    policy_net = DQN(grid_size=5, num_actions=3)

    try:
        policy_net.load_state_dict(torch.load("angry_birds_dqn.pth"))
        policy_net.eval() # Set PyTorch to evaluation mode (disables training features)
        print("Successfully loaded 'angry_birds_dqn.pth'")
    except FileNotFoundError:
        print("Error: Could not find the trained model file.")
        exit()

    # 3. Start the test episode
    state, _ = env.reset()
    print("\n=== AI Agent Navigating the Maze ===")
    env.render()

    done = False
    step_count = 0

    while not done:
        # Pause for half a second so we can visually track the sequence
        time.sleep(0.5)

        # Convert the numpy grid into a PyTorch tensor
        state_tensor = torch.FloatTensor(state).unsqueeze(0)

        # EXPLOIT: Ask the neural network for the best move (no random exploration)
        with torch.no_grad():
            q_values = policy_net(state_tensor)
            action = q_values.argmax().item()

        # Map action integer to text for our terminal output
        action_names = ["Move Forward", "Turn Left", "Turn Right"]
        print(f"Step {step_count + 1} | Agent chose: {action_names[action]}")

        # The environment reacts to the agent's choice
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        env.render()
        step_count += 1

        # Safety catch in case the agent learned a bad policy and loops forever
        if step_count > 20:
            print("Agent got stuck in a loop! Terminating early.")
            break

    # Print the final result
    if reward == 100:
        print("Success! The AI solved the sequence perfectly!")
    else:
        print(f"The AI failed (Reward: {reward}). It might need more training episodes or a tweaked reward function.")

Loading trained neural network...
Successfully loaded 'angry_birds_dqn.pth'

=== AI Agent Navigating the Maze ===
-----------
|. . . . v|
|# # # . .|
|. . . . .|
|. . # # #|
|P . . . .|
-----------


Step 1 | Agent chose: Move Forward
-----------
|. . . . .|
|# # # . v|
|. . . . .|
|. . # # #|
|P . . . .|
-----------


Step 2 | Agent chose: Move Forward
-----------
|. . . . .|
|# # # . .|
|. . . . v|
|. . # # #|
|P . . . .|
-----------


Step 3 | Agent chose: Turn Right
-----------
|. . . . .|
|# # # . .|
|. . . . <|
|. . # # #|
|P . . . .|
-----------


Step 4 | Agent chose: Move Forward
-----------
|. . . . .|
|# # # . .|
|. . . < .|
|. . # # #|
|P . . . .|
-----------


Step 5 | Agent chose: Move Forward
-----------
|. . . . .|
|# # # . .|
|. . < . .|
|. . # # #|
|P . . . .|
-----------


Step 6 | Agent chose: Move Forward
-----------
|. . . . .|
|# # # . .|
|. < . . .|
|. . # # #|
|P . . . .|
-----------


Step 7 | Agent chose: Turn Left
-----------
|. . . . .|
|# # # . .|
|. v . .

In [62]:
!pip install pygame

In [75]:
import pygame
import time
import torch
import numpy as np
import sys

# Import your classes from your original file
# Make sure to replace 'your_file_name' with the actual name of your python file
# from your_file_name import AngryBirdsGridEnv, DQN

# --- UI Configuration ---
WINDOW_SIZE = 500
GRID_SIZE = 5
CELL_SIZE = WINDOW_SIZE // GRID_SIZE

# Colors (R, G, B)
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
GRAY = (150, 150, 150) # Walls
RED = (200, 50, 50)    # Bird
GREEN = (50, 200, 50)  # Pig

def draw_grid(screen, grid, bird_dir):
    screen.fill(WHITE)

    for r in range(GRID_SIZE):
        for c in range(GRID_SIZE):
            val = grid[r, c]

            # Calculate pixel coordinates for this cell
            x = c * CELL_SIZE
            y = r * CELL_SIZE

            # Draw the cell background
            rect = pygame.Rect(x, y, CELL_SIZE, CELL_SIZE)
            pygame.draw.rect(screen, BLACK, rect, 1) # 1 = border thickness

            # Fill the cell based on what is in the matrix
            if val == 1:
                pygame.draw.rect(screen, GRAY, rect) # Wall
            elif val == 3:
                pygame.draw.rect(screen, GREEN, rect) # Pig
            elif val == 2:
                pygame.draw.rect(screen, RED, rect) # Bird

                # Draw a small line to show which way the bird is facing
                center_x, center_y = x + CELL_SIZE//2, y + CELL_SIZE//2
                dir_x, dir_y = center_x, center_y
                if bird_dir == 0: dir_y -= 20 # North
                elif bird_dir == 1: dir_x += 20 # East
                elif bird_dir == 2: dir_y += 20 # South
                elif bird_dir == 3: dir_x -= 20 # West
                pygame.draw.line(screen, BLACK, (center_x, center_y), (dir_x, dir_y), 3)

    pygame.display.flip() # Push updates to the screen

if __name__ == "__main__":
    # 1. Initialize Pygame
    pygame.init()
    screen = pygame.display.set_mode((WINDOW_SIZE, WINDOW_SIZE))
    pygame.display.set_caption("Agentic System Observer: Angry Birds")

    # 2. Load the Environment and Model
    env = AngryBirdsGridEnv(render_mode=None) # We handle rendering here now
    policy_net = DQN(grid_size=GRID_SIZE, num_actions=3)

    try:
        policy_net.load_state_dict(torch.load("angry_birds_dqn.pth"))
        policy_net.eval()
        print("Model loaded successfully.")
    except:
        print("Error loading model. Make sure 'angry_birds_dqn.pth' exists.")
        sys.exit()

    # 3. The Main Game/Inference Loop
    state, _ = env.reset()
    done = False

    while not done:
        # Pygame event pump (keeps the window from freezing in your OS)
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()

        # Draw the current state
        draw_grid(screen, env.grid, env.bird_dir)
        time.sleep(0.5) # Pause so you can watch it

        # Ask the AI for the next move
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        with torch.no_grad():
            q_values = policy_net(state_tensor)
            action = q_values.argmax().item()

        # Step the environment
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

    # Draw the final frame
    draw_grid(screen, env.grid, env.bird_dir)
    print(f"Episode Finished. Reward: {reward}")
    time.sleep(2) # Keep window open for 2 seconds to see result
    pygame.quit()

Model loaded successfully.
Episode Finished. Reward: 100


In [80]:
try:
    bird_img = pygame.image.load('bird.png')
    bird_img = pygame.transform.scale(bird_img, (CELL_SIZE, CELL_SIZE))

    pig_img = pygame.image.load('pig.png')
    pig_img = pygame.transform.scale(pig_img, (CELL_SIZE, CELL_SIZE))

    wall_img = pygame.image.load('wall.png')
    wall_img = pygame.transform.scale(wall_img, (CELL_SIZE, CELL_SIZE))
except FileNotFoundError:
    print("Warning: Could not find image files. Please upload bird.png, pig.png, and wall.png to Colab.")


In [103]:
import os
# CRITICAL: Tell Pygame to use a dummy video driver since Colab has no monitor
os.environ["SDL_VIDEODRIVER"] = "dummy"

import pygame
import torch
import numpy as np
import time
import matplotlib.pyplot as plt
from IPython import display

# --- UI Configuration ---
WINDOW_SIZE = 500
GRID_SIZE = 5
CELL_SIZE = WINDOW_SIZE // GRID_SIZE

# Colors (R, G, B)
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
GRAY = (150, 150, 150) # Walls
RED = (200, 50, 50)    # Bird
GREEN = (50, 200, 50)  # Pig

# Load images and scale them to fit the grid cells

def draw_grid_colab(screen, grid, bird_dir):
    screen.fill(WHITE) # Keep the white background

    for r in range(GRID_SIZE):
        for c in range(GRID_SIZE):
            val = grid[r, c]
            x = c * CELL_SIZE
            y = r * CELL_SIZE

            # Optional: Keep a faint grid line for structure
            rect = pygame.Rect(x, y, CELL_SIZE, CELL_SIZE)
            pygame.draw.rect(screen, GRAY, rect, 1)

            if val == 1:
                # Draw the wall image
              screen.blit(wall_img, (x, y))
            elif val == 3:

                # Draw the pig image
              screen.blit(pig_img, (x, y))
            elif val == 2:

                # Rotate the bird based on its direction
                # 0: North, 1: East, 2: South, 3: West
              angle = 0
              if bird_dir == 1: angle = -90   # Turn right
              elif bird_dir == 2: angle = 180 # Turn around
              elif bird_dir == 3: angle = 90  # Turn left

              rotated_bird = pygame.transform.rotate(bird_img, angle)
              screen.blit(rotated_bird, (x, y))

    # Extract the pixel array for Colab exactly like before
    view = pygame.surfarray.array3d(screen)
    view = view.transpose([1, 0, 2])
    return view

# 1. Initialize Dummy Pygame
pygame.init()
screen = pygame.display.set_mode((WINDOW_SIZE, WINDOW_SIZE))

# 2. Load the Environment and Model
env = AngryBirdsGridEnv(render_mode=None)
policy_net = DQN(grid_size=GRID_SIZE, num_actions=3)

try:
    policy_net.load_state_dict(torch.load("angry_birds_dqn.pth"))
    policy_net.eval()
    print("Model loaded successfully.")
except:
    print("Error: Could not find 'angry_birds_dqn.pth'. Make sure you trained the model in this Colab session!")

# 3. The Inference Loop for Colab
state, _ = env.reset()
done = False
step_count = 0

# Set up matplotlib figure
fig, ax = plt.subplots(figsize=(5, 5))

while not done:
    # Get pixels and render in Colab
    img_array = draw_grid_colab(screen, env.grid, env.bird_dir)
    ax.clear()
    ax.imshow(img_array)
    ax.axis('off')

    # Refresh the cell output to create an animation
    display.display(fig)
    display.clear_output(wait=True)
    time.sleep(0.5)

    # AI Decision Making
    state_tensor = torch.FloatTensor(state).unsqueeze(0)
    with torch.no_grad():
        q_values = policy_net(state_tensor)
        action = q_values.argmax().item()

    # Environment Step
    state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    step_count += 1

    if step_count > 20:
        break

img_array = draw_grid_colab(screen, env.grid, env.bird_dir)
ax.clear()
ax.imshow(img_array)
ax.axis('off')
display.display(fig)
display.clear_output(wait=True)

print(f"Episode Finished in {step_count} steps. Total Reward: {reward}")
plt.close(fig)

Episode Finished in 11 steps. Total Reward: 100
